# MetroGuard AI - Notebook 01: Exploratory Data Analysis & Sensor Validation

**Course**: ML Mini Project · 3 Students · October 2026  
**Dataset**: MetroPT-3 (Air Production Unit, Metro do Porto) · UCI ML Repository / Nature Scientific Data (2022)  
**Goal**: Validate temporal ordering, audit sensor data quality, examine physical distributions, and analyze documented air leak failure intervals.

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in python path
sys.path.append(os.path.abspath('..'))
from src.data_validation import load_config, validate_raw_telematics, generate_eda_figures

config = load_config('../config/config.yaml')
print('Config loaded successfully. Seed:', config['project']['random_seed'])

## 1. Load and Inspect Raw Telematics
We load the unadulterated dataset from `../data/raw/MetroPT3(AirCompressor).csv` without modifying the original source file.

In [ ]:
raw_csv_path = os.path.join('..', config['paths']['raw_data'])
print(f'Reading {raw_csv_path}...')
df_raw = pd.read_csv(raw_csv_path)
print('Shape:', df_raw.shape)
df_raw.head()

## 2. Execute Data Quality & Integrity Validation
Here we execute `validate_raw_telematics()` to confirm:
1. Chronological order & monotonicity
2. Timestamp duplicate counts
3. Missing values per sensor
4. Physical bounds for continuous sensors
5. Sampling intervals & downtime gaps

In [ ]:
df_clean, report = validate_raw_telematics(df_raw, config)
import json
print(json.dumps(report['sampling_analysis'], indent=2))
print('\nMissing values summary:', sum(report['missing_values'].values()))
print('Duplicate timestamps:', report['duplicate_timestamps'])

## 3. Sensor Telematics & Failure Event Visualization
We plot pressure (`TP2`, `Reservoirs`), oil temperature, and motor current across the 7-month operational period, shading the four official air leak failure events in red.

In [ ]:
generate_eda_figures(df_clean, config, output_dir='../reports/figures')
print('Generated figures in reports/figures/:')
for f in os.listdir('../reports/figures'):
    print('-', f)

## 4. Key Findings and Defensibility Summary
- **Zero Null Values**: The MetroPT-3 embedded logger recorded 1,516,948 samples without any missing numerical values.
- **Strict Chronological Monotonicity**: Timestamps increase strictly at ~10-second intervals (median step = 10s).
- **Documented Failures**: 4 high-stress Air Leak incidents reported by Metro do Porto operators (April 18, May 29-30, June 5-7, July 15, 2020).
- **Sensor Offsets**: Slight negative values in TP2 (down to -0.014 bar) reflect standard industrial transducer zero-point calibration drift under atmospheric resting conditions and must NOT be deleted.